# 01 - Load the dataset

**What this does:** loads `data/processed/posts.parquet` — the ONE merged
dataset (7.95M Reddit posts across 15 subreddits 2008-2025, plus ~2.9M X
(Twitter) finance posts covering 2015-2020 and 2022-2024 — note the hole
across 2021 — once `add_x_data.py` has been run; every row carries a
`source` column, 'reddit' or 'x') — and lets you slice it by subreddit and
date without touching the raw files.

**This notebook is the head of the chain.** It saves the filtered slice to
`data/processed/posts_slice.parquet`, and that file is what notebooks
02 (ticker mentions) and 04 (theme mentions) read — so the TIME WINDOW and
SUBREDDITS you set below apply to the whole chain (01 → 02 → 03 and
01 → 04 → 05) without re-typing them anywhere. It also builds the
word-ticker screening table (last section) that notebook 02's extractor
uses to ignore words like LOAN / EDGE / RENT unless written as $cashtags.

**This notebook no longer reads `.zst` files.** The raw-to-parquet conversion
already happened (it's slow — ~3 GB of compressed JSON) and lives in
`data_ingestion/scripts/prep_posts.py` (Reddit) and
`data_ingestion/scripts/add_x_data.py` (X). You only re-run those if the raw
dumps change or you want different cleaning rules. Day-to-day analysis
starts HERE, from the parquet, which loads in seconds.

Because the parquet is *columnar* and stored in subreddit blocks (X rows are
their own `x_twitter` block), we can read just the columns and subreddits we
need instead of the whole 1.15 GB file.

In [1]:
import os, sys
# Find the project root so we can import the src/ package and locate data/.
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
print('project root:', ROOT)

project root: C:\Users\alexd\Desktop\GIC\RetailFlow1


In [2]:
# ============ TIME WINDOW ============
# THE single place the analysis window is set: notebooks 02/03/04/05 all
# inherit it through the slice file saved at the bottom of this notebook.
#
# LIVE RUNS: update_data.py sets the window through environment variables so
# it extends automatically as new data arrives - you never edit this cell
# for a daily refresh:
#     PIPELINE_START_DATE  history depth, 'YYYY-MM-DD' inclusive
#     PIPELINE_END_DATE    'YYYY-MM-DD' EXCLUSIVE, or empty/'None' = open
#                          (open => the window always reaches the newest post)
#
# BY HAND (Jupyter, no env vars): the fallbacks below are used instead -
# edit them freely for ad-hoc analysis. Keep the window narrow to keep runs
# fast (all 15 subs: 1 year ~ 700k-2.8M posts; all dates ~ 7.9M -> 02 slow).
# NOTE: the X (Twitter) history has a HOLE across all of 2021.
START_DATE = os.environ.get('PIPELINE_START_DATE') or '2023-10-01'   # inclusive
_end = (os.environ.get('PIPELINE_END_DATE') or '').strip()
END_DATE = _end if _end and _end.lower() != 'none' else None         # None = open
print(f'TIME WINDOW: START_DATE={START_DATE}  END_DATE={END_DATE}'
      f"  (source: {'run_daily/env' if os.environ.get('PIPELINE_START_DATE') else 'notebook fallback'})")
# ====================================================

TIME WINDOW: START_DATE=2017-01-01  END_DATE=None  (source: run_daily/env)


In [3]:
# ============ PARAMETERS - edit these ============
POSTS_PATH = os.path.join(ROOT, 'data', 'processed', 'posts.parquet')
SUBREDDITS = []      # e.g. ['wallstreetbets', 'stocks'];  [] = ALL (incl. 'x_twitter'
                     # once X data is merged - list it like a subreddit to isolate it)
COLUMNS    = None    # keep None when saving the slice - 02/04 need all columns
SLICE_OUT  = os.path.join(ROOT, 'data', 'processed', 'posts_slice.parquet')
                     # notebooks 02 and 04 READ this file - it is how they
                     # inherit the window above. Set to None to skip saving.
# ==================================================

In [4]:
# Quick metadata peek - reads only the file FOOTER, so it is instant
# even though the file is 1.15 GB.
import pyarrow.parquet as pq

pf = pq.ParquetFile(POSTS_PATH)
print('rows       :', f'{pf.metadata.num_rows:,}')
print('columns    :', pf.schema_arrow.names)
print('row groups :', pf.metadata.num_row_groups)

rows       : 8,248,511
columns    : ['id', 'date', 'author', 'score', 'subreddit', 'title', 'selftext', 'num_comments', 'source']
row groups : 51


In [5]:
# Load the data - with filters pushed down into the parquet reader.
# pyarrow uses each row group's min/max stats to SKIP blocks that can't
# match, so asking for one subreddit reads ~1/30th of the file.

### IMPT: This takes a while but works

import pandas as pd

filters = []
if SUBREDDITS:
    filters.append(('subreddit', 'in', [s.lower() for s in SUBREDDITS]))
if START_DATE:
    filters.append(('date', '>=', START_DATE))
if END_DATE:
    filters.append(('date', '<', END_DATE))

posts = pq.read_table(
    POSTS_PATH,
    columns=COLUMNS,
    filters=filters if filters else None,
).to_pandas()

print('loaded', f'{len(posts):,}', 'posts')
print()
print(posts['subreddit'].value_counts())

# Where did the rows come from? (source = 'reddit' or 'x')
print()
if 'source' in posts.columns:
    print('--- sources in this window ---')
    print(posts.groupby('source')['date'].agg(['count', 'min', 'max']))
else:
    print("(no 'source' column - X data not merged yet; run",
          'data_ingestion/scripts/fetch_x_data.py then add_x_data.py)')

loaded 7,224,423 posts



subreddit
wallstreetbets           2276218
cryptocurrency           1717631
personalfinance          1082343
bitcoin                   801606
stocks                    308496
investing                 254407
pennystocks               200001
stockmarket               135474
daytrading                120815
options                   109863
financialindependence      74368
dividends                  42668
thetagang                  35533
valueinvesting             29868
securityanalysis           15399
bogleheads                  8003
stocktwits                  6261
x_twitter                   3267
finance                     2202
Name: count, dtype: int64

--- sources in this window ---


              count         min         max
source                                     
reddit      7214895  2017-01-01  2026-07-18
stocktwits     6261  2023-12-03  2026-07-18
x              3267  2017-06-14  2026-07-18


In [6]:
# Preview.
posts.head()

,id,date,author,score,subreddit,title,selftext,num_comments,source
0,5lch52,2017-01-01,[deleted],2,bitcoin,Do I need to backup my wallet every time my bi...,[deleted],17,reddit
1,5lch6f,2017-01-01,mr_robot-sh,7,bitcoin,How do you put a message in a block on the blo...,I remember reading somewhere about people putt...,14,reddit
2,5lcion,2017-01-01,LolzNathan,8,bitcoin,Advice: Coinbase not worth touching due to the...,,45,reddit
3,5lcjms,2017-01-01,BitcoinRush,19,bitcoin,"Will BITCOIN put to shame Gold price (1 oc $1,...",,13,reddit
4,5lckn6,2017-01-01,Tony_Traveler,4,bitcoin,Buying Bitcoin,I live in the US and I've been buying bitcoin ...,5,reddit


In [7]:
# Save the slice. Notebooks 02 (tickers) and 04 (themes) read THIS file,
# so they automatically inherit the TIME WINDOW + SUBREDDITS set above -
# change the window here, re-run this notebook, and the chain follows.
if SLICE_OUT:
    posts.to_parquet(SLICE_OUT, index=False)
    print('saved', f'{len(posts):,}', 'posts ->', SLICE_OUT)
    print('window in file:', posts['date'].min(), '->', posts['date'].max())
else:
    print('SLICE_OUT is None - nothing saved. Notebooks 02/04 NEED this file;')
    print('set SLICE_OUT in the PARAMETERS cell and re-run.')

saved 7,224,423 posts -> C:\Users\alexd\Desktop\GIC\RetailFlow1\data\processed\posts_slice.parquet


window in file: 2017-01-01 -> 2026-07-18


## Screen word-tickers - case ratio + wordfreq

Some real tickers are also everyday English words (EDGE, LOAN, RENT, TECH,
EARN...). Bare-caps matching counts those words as phantom mentions, and a
hand-maintained stop list never ends. So we let the data decide, using two
signals:

1. **Case ratio (our own corpus).** English words appear mostly lowercase in
   real posts ("the edge of"); tickers appear mostly ALL-CAPS ("NVDA calls").
   For each candidate we count both forms and compute
   `caps_share = caps / (caps + lower)`.
   Measured on this dataset: EDGE ~ 0.02, LOAN ~ 0.002, NVDA ~ 0.93.
2. **wordfreq (fallback).** If a candidate has fewer than 30 sightings here,
   the ratio is not trustworthy, so we ask how common its lowercase form is
   in general English (zipf scale: "edge" 4.7 = common word, "nvda" 2.1 = not).

**Corpus evidence wins over wordfreq**: SNAP looks like a word to wordfreq
(zipf 4.2) but the corpus measures caps_share ~ 0.56 - people write SNAP
like a ticker - so it stays. That is why signal 1 is checked first.

Word-tickers are **demoted, not deleted**: they still count when written as
a cashtag ($LOAN), so a real attention spike is never lost - only the prose
noise is. `src/extract_tickers.py` loads the CSV automatically at import;
notebook 02 needs no changes.

Caveats (details in README "Screening word-tickers"): caps-typed jargon
(HODL) passes the case test - the manual stop list still covers those; and
brand-name tickers people type lowercase (SOFI, HOOD, COIN) get demoted -
that trades recall for precision, the right trade before Stage 2.

In [8]:
# ============ SCREENING PARAMETERS - edit freely ============
SCREEN_SAMPLE_SIZE = 300_000   # posts used for the case-ratio count (~40 s)
CLASSIFICATION_OUT = os.path.join(ROOT, 'data', 'reference', 'ticker_classification.csv')
# The thresholds (MIN_SIGHTINGS=30, CAPS_SHARE_THRESHOLD=0.5,
# ZIPF_WORD_THRESHOLD=3.5) live in src/screen_tickers.py - tune them there
# if the tables below look wrong, then re-run this section.
# ============================================================

In [9]:
# Classify every 4-5 letter ticker in the universe (only those can collide
# with the bare-caps regex WORD_BARE). Uses the posts loaded above, so the
# ratio is measured on the SAME time window / subreddits you are analysing
# (X rows, if present in the window, are part of the sample too).
from pathlib import Path
from src.screen_tickers import screen_tickers
from src.ticker_universe import load_us_ticker_universe

universe = load_us_ticker_universe(
    Path(ROOT) / 'data' / 'reference',  # cached Nasdaq symbol files
    max_cache_age_days=365,             # use the cache; avoids re-downloading
)
candidates = {t for t in universe if 4 <= len(t) <= 5}
print('candidate tickers to screen:', len(candidates))

# A sample is plenty - 300k posts already sights common words thousands
# of times. random_state=0 makes the sample (and the CSV) reproducible.
sample = posts.sample(min(SCREEN_SAMPLE_SIZE, len(posts)), random_state=0)
texts = (sample['title'].fillna('') + ' ' + sample['selftext'].fillna('')).tolist()

classification = screen_tickers(texts, candidates)   # ~40 s for 300k posts
classification.to_csv(CLASSIFICATION_OUT, index=False)

n_demoted = (classification['classification'] == 'cashtag_only').sum()
print('saved ->', CLASSIFICATION_OUT)
print(n_demoted, 'tickers demoted to cashtag-only |',
      len(classification) - n_demoted, 'kept normal')

candidate tickers to screen: 9349


saved -> C:\Users\alexd\Desktop\GIC\RetailFlow1\data\reference\ticker_classification.csv
433 tickers demoted to cashtag-only | 8916 kept normal


In [10]:
# Sanity-check the decisions.
# Table 1: worst word-ticker offenders we just demoted - a huge lower_count
# means the word is everywhere in prose, exactly what was polluting counts.
seen = classification[classification['caps_count'] + classification['lower_count'] >= 30].copy()
demoted = seen[seen['classification'] == 'cashtag_only']
print('--- top demoted word-tickers ---')
print(demoted.sort_values('lower_count', ascending=False).head(15).to_string(index=False))

# Table 2: the borderline zone - the only rows worth eyeballing by hand.
# If one looks wrong, add it to the manual stop lists in extract_tickers.py
# (definitely a word) or leave it - its $cashtag mentions always count.
borderline = seen[(seen['caps_share'] > 0.35) & (seen['caps_share'] < 0.65)].copy()
borderline['total'] = borderline['caps_count'] + borderline['lower_count']
print()
print('--- borderline (0.35 < caps_share < 0.65), most-seen first ---')
print(borderline.sort_values('total', ascending=False).head(15).to_string(index=False))

--- top demoted word-tickers ---
ticker  caps_count  lower_count  caps_share  zipf decided_by classification
  JUST         308        46451       0.007  6.43 case_ratio   cashtag_only
  TIME         662        31293       0.021  6.29 case_ratio   cashtag_only
  YEAR          76        31229       0.002  5.96 case_ratio   cashtag_only
  KNOW         126        25343       0.005  6.10 case_ratio   cashtag_only
  HERE        3479        21999       0.137  5.97 case_ratio   cashtag_only
  WANT         127        20134       0.006  6.04 case_ratio   cashtag_only
  GOOD         128        20048       0.006  6.12 case_ratio   cashtag_only
  HELP         436        16868       0.025  5.75 case_ratio   cashtag_only
  NEXT         368        15212       0.024  5.70 case_ratio   cashtag_only
  CARD          30        13964       0.002  5.04 case_ratio   cashtag_only
  LOAN          36        13548       0.003  4.68 case_ratio   cashtag_only
  WEEK         546        12702       0.041  5.56 case_